<a href="https://colab.research.google.com/github/avocado-planet/03-Langchain_Middleware/blob/main/03_langchain_middleware.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧩 LangChain 1.0 ミドルウェア 完全ガイド

このノートブックでは、LangChain 1.0 で導入された **ミドルウェア（Middleware）** の全パターンを
実際に動くコードで学びます。

## 目次
1. **セットアップ** — パッケージインストール・APIキー設定
2. **基本概念** — ミドルウェアとは？エージェントループの仕組み
3. **ノードスタイルフック** — `before_agent`, `before_model`, `after_model`, `after_agent`
4. **ラップスタイルフック** — `wrap_model_call`, `wrap_tool_call`
5. **デコレータ形式 vs クラス形式** — 2つの書き方
6. **カスタムState** — ミドルウェア専用のステートを追加
7. **実行順序** — 複数ミドルウェアの実行フロー
8. **Agent Jumps** — `jump_to` でフロー制御
9. **`dynamic_prompt`** — 動的にシステムプロンプトを生成
10. **`wrap_model_call` 応用** — モデル切替・システムメッセージ加工
11. **ビルトインミドルウェア** — SummarizationMiddleware 等
12. **総合例** — 複数ミドルウェアを組み合わせた本番想定パターン

---
## 1. セットアップ

In [18]:
# LangChain 1.0 と LangGraph をインストール
# --pre フラグで最新のアルファ/ベータ版を取得
!pip install --pre -U langchain langgraph langchain-openai langchain-anthropic -q

In [19]:
import os
from getpass import getpass
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# ============================================
# APIキーの設定（OpenAI or Anthropic）
# Colabの場合は Secrets機能 を使うのが安全です
# ============================================

# --- OpenAI を使う場合 ---
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

# --- Anthropic を使う場合（コメント解除して使用） ---
# if "ANTHROPIC_API_KEY" not in os.environ:
#     os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API Key: ")

print("✅ APIキー設定完了")

✅ APIキー設定完了


In [20]:
# ============================================
# 使用するモデルの設定
# お手元の環境に合わせて変更してください
# ============================================

# OpenAI を使う場合
MODEL_NAME = "openai:gpt-4.1-mini"  # 安価で高速
MODEL_NAME_CHEAP = "openai:gpt-4.1-nano"  # さらに安価（要約用）

# Anthropic を使う場合（コメント解除）
# MODEL_NAME = "anthropic:claude-sonnet-4-20250514"
# MODEL_NAME_CHEAP = "anthropic:claude-haiku-4-5-20251001"

print(f"メインモデル: {MODEL_NAME}")
print(f"軽量モデル: {MODEL_NAME_CHEAP}")

メインモデル: openai:gpt-4.1-mini
軽量モデル: openai:gpt-4.1-nano


In [21]:
# ============================================
# 共通で使うツール（道具）を定義
# エージェントが呼び出せる関数です
# ============================================
from langchain_core.tools import tool
import random


@tool
def get_weather(city: str) -> str:
    """指定した都市の天気を取得します。"""
    weathers = ["晴れ ☀️", "曇り ☁️", "雨 🌧️", "雪 ❄️"]
    temp = random.randint(-5, 35)
    return f"{city}の天気: {random.choice(weathers)}, 気温: {temp}°C"


@tool
def calculator(expression: str) -> str:
    """数式を計算します。例: '2 + 3 * 4'"""
    try:
        result = eval(expression)  # デモ用。本番では安全な評価器を使用
        return f"計算結果: {expression} = {result}"
    except Exception as e:
        return f"計算エラー: {e}"


@tool
def search_web(query: str) -> str:
    """Webを検索します（デモ用のモック）。"""
    return f"検索結果: '{query}' に関する情報が見つかりました。（これはモックデータです）"


# ツール一覧
ALL_TOOLS = [get_weather, calculator, search_web]

print("✅ ツール定義完了")
for t in ALL_TOOLS:
    print(f"  - {t.name}: {t.description}")

✅ ツール定義完了
  - get_weather: 指定した都市の天気を取得します。
  - calculator: 数式を計算します。例: '2 + 3 * 4'
  - search_web: Webを検索します（デモ用のモック）。


---
## 2. 基本概念 — ミドルウェアとは？

### エージェントのコアループ

LangChain のエージェントは以下のループで動きます:

```
ユーザー入力
    ↓
┌─── エージェントループ ───┐
│  モデル呼び出し          │
│    ↓                    │
│  ツール呼び出し?         │
│    YES → ツール実行      │
│    NO  → 完了            │
└─────────────────────────┘
    ↓
最終応答
```

### ミドルウェアの介入ポイント

ミドルウェアはこのループの **各段階** にフック（介入）できます:

```
before_agent  ← エージェント開始前（1回だけ）
    ↓
┌─── ループ ──────────────────┐
│  before_model  ← 毎回       │
│    ↓                        │
│  wrap_model_call ← モデル包む│
│    ↓                        │
│  after_model   ← 毎回       │
│    ↓                        │
│  wrap_tool_call ← ツール包む │
└─────────────────────────────┘
    ↓
after_agent   ← エージェント完了後（1回だけ）
```

---
## 3. ノードスタイルフック

特定のタイミングで **順番に** 実行されるフックです。
`dict` を返すとエージェントの State にマージされます。

### 3-1. `@before_model` — モデル呼び出し前

In [22]:
from langchain.agents import create_agent
from langchain.agents.middleware import before_model, AgentState
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime
from typing import Any


# ============================================
# @before_model デコレータ
# モデルが呼ばれる「直前」に毎回実行される
# ============================================
@before_model
def log_before_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """モデル呼び出し前にメッセージ数をログ出力する。"""
    msg_count = len(state["messages"])
    print(f"📥 [before_model] メッセージ数: {msg_count}")

    # 最新メッセージの内容を表示
    last_msg = state["messages"][-1]
    print(f"   最新メッセージ: {type(last_msg).__name__} → {str(last_msg.content)[:80]}")

    # None を返す = State を変更しない
    return None


# エージェント作成
agent_1 = create_agent(
    model=MODEL_NAME,
    tools=[get_weather],
    middleware=[log_before_model],
)

# 実行
print("=" * 60)
print("🚀 エージェント実行開始")
print("=" * 60)
result = agent_1.invoke({"messages": [HumanMessage("東京の天気を教えて")]})
print("\n" + "=" * 60)
print("✅ 最終応答:")
print(result["messages"][-1].content)

🚀 エージェント実行開始
📥 [before_model] メッセージ数: 1
   最新メッセージ: HumanMessage → 東京の天気を教えて
📥 [before_model] メッセージ数: 3
   最新メッセージ: ToolMessage → 東京の天気: 雨 🌧️, 気温: 28°C

✅ 最終応答:
現在の東京の天気は雨で、気温は28度です。ほかに知りたいことはありますか？


### 3-2. `@after_model` — モデル呼び出し後

In [23]:
from langchain.agents.middleware import after_model


# ============================================
# @after_model デコレータ
# モデルの応答が返ってきた「直後」に毎回実行される
# ============================================
@after_model
def log_after_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """モデル応答後に、応答内容をログ出力する。"""
    last_msg = state["messages"][-1]
    content = str(last_msg.content)[:100]
    print(f"📤 [after_model] 応答: {content}")

    # ツール呼び出しがある場合はそれも表示
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        for tc in last_msg.tool_calls:
            print(f"   🔧 ツール呼び出し: {tc['name']}({tc['args']})")

    return None


# before + after を組み合わせる
agent_2 = create_agent(
    model=MODEL_NAME,
    tools=[get_weather],
    middleware=[log_before_model, log_after_model],
)

print("=" * 60)
print("🚀 before + after 両方を使ったエージェント")
print("=" * 60)
result = agent_2.invoke({"messages": [HumanMessage("大阪の天気は？")]})
print("\n✅ 最終応答:", result["messages"][-1].content)

🚀 before + after 両方を使ったエージェント
📥 [before_model] メッセージ数: 1
   最新メッセージ: HumanMessage → 大阪の天気は？
📤 [after_model] 応答: 
   🔧 ツール呼び出し: get_weather({'city': '大阪'})
📥 [before_model] メッセージ数: 3
   最新メッセージ: ToolMessage → 大阪の天気: 晴れ ☀️, 気温: 5°C
📤 [after_model] 応答: 大阪の天気は晴れで、気温は5°Cです。何か他に知りたいことはありますか？

✅ 最終応答: 大阪の天気は晴れで、気温は5°Cです。何か他に知りたいことはありますか？


### 3-3. `@before_agent` / `@after_agent` — エージェント開始前・完了後

ループ全体の **前後に1回だけ** 実行されます。  
初期化処理やクリーンアップに便利です。

In [24]:
from langchain.agents.middleware import before_agent, after_agent
import time


# ============================================
# @before_agent — エージェント実行開始時に1回だけ
# ============================================
@before_agent
def on_agent_start(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """エージェント開始時に実行。タイムスタンプを記録。"""
    print(f"🟢 [before_agent] エージェント開始! メッセージ数: {len(state['messages'])}")
    # NOTE: 時間計測などの副作用はここで行う
    return None


# ============================================
# @after_agent — エージェント実行完了時に1回だけ
# ============================================
@after_agent
def on_agent_end(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """エージェント完了時に実行。最終メッセージ数を表示。"""
    print(f"🔴 [after_agent] エージェント完了! 最終メッセージ数: {len(state['messages'])}")
    return None


# 4つのフック全部を組み合わせ
agent_3 = create_agent(
    model=MODEL_NAME,
    tools=[get_weather],
    middleware=[on_agent_start, log_before_model, log_after_model, on_agent_end],
)

print("=" * 60)
print("🚀 4つのノードスタイルフック全部入り")
print("=" * 60)
result = agent_3.invoke({"messages": [HumanMessage("札幌の天気を調べて")]})
print("\n✅ 最終応答:", result["messages"][-1].content)

🚀 4つのノードスタイルフック全部入り
🟢 [before_agent] エージェント開始! メッセージ数: 1
📥 [before_model] メッセージ数: 1
   最新メッセージ: HumanMessage → 札幌の天気を調べて
📤 [after_model] 応答: 
   🔧 ツール呼び出し: get_weather({'city': '札幌'})
📥 [before_model] メッセージ数: 3
   最新メッセージ: ToolMessage → 札幌の天気: 曇り ☁️, 気温: 7°C
📤 [after_model] 応答: 札幌の天気は曇りで、気温は7°Cです。
🔴 [after_agent] エージェント完了! 最終メッセージ数: 4

✅ 最終応答: 札幌の天気は曇りで、気温は7°Cです。


---
## 4. ラップスタイルフック

モデル呼び出しやツール実行を **包み込む** フックです。  
リトライ、キャッシュ、変換など **呼び出し自体を制御** できます。

### 4-1. `@wrap_model_call` — モデル呼び出しをラップ

In [25]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable
import time


# ============================================
# @wrap_model_call
# handler を呼ぶ/呼ばない を自分で決められる
# → リトライ、計測、キャッシュ等に使える
# ============================================
@wrap_model_call
def measure_model_time(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """モデル呼び出しの所要時間を計測する。"""
    start = time.time()
    print(f"⏱️ [wrap_model_call] モデル呼び出し開始...")

    # handler を呼ぶ = 実際のモデル呼び出し
    response = handler(request)

    elapsed = time.time() - start
    print(f"⏱️ [wrap_model_call] 完了! 所要時間: {elapsed:.2f}秒")
    return response


agent_4 = create_agent(
    model=MODEL_NAME,
    tools=[get_weather],
    middleware=[measure_model_time],
)

print("=" * 60)
print("🚀 wrap_model_call でモデル呼び出し時間を計測")
print("=" * 60)
result = agent_4.invoke({"messages": [HumanMessage("福岡の天気は？")]})
print("\n✅ 最終応答:", result["messages"][-1].content)

🚀 wrap_model_call でモデル呼び出し時間を計測
⏱️ [wrap_model_call] モデル呼び出し開始...
⏱️ [wrap_model_call] 完了! 所要時間: 0.98秒
⏱️ [wrap_model_call] モデル呼び出し開始...
⏱️ [wrap_model_call] 完了! 所要時間: 0.69秒

✅ 最終応答: 福岡の天気は雪で、気温は9°Cです。お出かけの際は暖かくしてお出かけください。ほかに知りたいことはありますか？


### 4-2. `@wrap_model_call` — リトライ（再試行）パターン

In [26]:
# ============================================
# リトライパターン
# handler を複数回呼べる = リトライが可能
# ============================================
@wrap_model_call
def retry_on_error(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """エラー時に最大3回リトライする。"""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = handler(request)
            if attempt > 0:
                print(f"🔄 [retry] {attempt + 1}回目で成功!")
            return response
        except Exception as e:
            print(f"🔄 [retry] 試行 {attempt + 1}/{max_retries} 失敗: {e}")
            if attempt == max_retries - 1:
                raise  # 最終試行でも失敗したら例外を投げる
            time.sleep(1)  # 少し待ってからリトライ


agent_5 = create_agent(
    model=MODEL_NAME,
    tools=[get_weather],
    middleware=[retry_on_error],
)

print("=" * 60)
print("🚀 リトライミドルウェア（正常時はそのまま通過）")
print("=" * 60)
result = agent_5.invoke({"messages": [HumanMessage("名古屋の天気は？")]})
print("✅ 最終応答:", result["messages"][-1].content)

🚀 リトライミドルウェア（正常時はそのまま通過）
✅ 最終応答: 名古屋の現在の天気は曇りで、気温は24度です。


### 4-3. `@wrap_tool_call` — ツール呼び出しをラップ

In [27]:
from langchain.agents.middleware import wrap_tool_call
from langchain.tools.tool_node import ToolCallRequest
from langchain.messages import ToolMessage
from langgraph.types import Command


# ============================================
# @wrap_tool_call
# ツール実行を包み込む。ログ、計測、ガード等に使う
# ============================================
@wrap_tool_call
def monitor_tool_call(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage | Command],
) -> ToolMessage | Command:
    """ツール呼び出しを監視し、実行前後にログを出す。"""
    tool_name = request.tool_call["name"]
    tool_args = request.tool_call["args"]
    print(f"🔧 [wrap_tool_call] ツール実行開始: {tool_name}({tool_args})")

    start = time.time()
    result = handler(request)  # 実際のツール実行
    elapsed = time.time() - start

    print(f"🔧 [wrap_tool_call] ツール完了: {elapsed:.3f}秒")
    if isinstance(result, ToolMessage):
        print(f"   結果: {str(result.content)[:80]}")

    return result


agent_6 = create_agent(
    model=MODEL_NAME,
    tools=[get_weather, calculator],
    middleware=[monitor_tool_call],
)

print("=" * 60)
print("🚀 wrap_tool_call でツール実行を監視")
print("=" * 60)
result = agent_6.invoke({"messages": [HumanMessage("京都の天気と、123 * 456 を計算して")]})
#result = agent_6.invoke({"messages": [HumanMessage("komae cityはどこにありますか")]})
print("\n✅ 最終応答:", result["messages"][-1].content)

🚀 wrap_tool_call でツール実行を監視
🔧 [wrap_tool_call] ツール実行開始: get_weather({'city': '京都'})
🔧 [wrap_tool_call] ツール完了: 0.001秒
   結果: 京都の天気: 雨 🌧️, 気温: 32°C
🔧 [wrap_tool_call] ツール実行開始: calculator({'expression': '123 * 456'})
🔧 [wrap_tool_call] ツール完了: 0.001秒
   結果: 計算結果: 123 * 456 = 56088

✅ 最終応答: 京都の天気は雨で、気温は32°Cです。
また、123 * 456 の計算結果は56088です。


---
## 5. デコレータ形式 vs クラス形式

### 使い分けガイド

| 特徴 | デコレータ形式 | クラス形式 |
|---|---|---|
| 適用場面 | 単機能・シンプル | 複雑・複数フック |
| 設定(Config) | 関数の外で変数参照 | `__init__` で渡す |
| 同期/非同期 | 1つだけ定義 | sync/async 両方定義可 |
| 再利用性 | 低い | 高い |

In [28]:
from langchain.agents.middleware import AgentMiddleware


# ============================================
# クラス形式: 複数フック + 設定をまとめられる
# ============================================
class FullLoggingMiddleware(AgentMiddleware):
    """エージェントの全ライフサイクルをロギングするミドルウェア。"""

    def __init__(self, verbose: bool = True):
        super().__init__()
        self.verbose = verbose
        self.call_count = 0

    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print("🟢 [CLASS] エージェント開始")
        self.call_count = 0
        return None

    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        self.call_count += 1
        if self.verbose:
            print(f"📥 [CLASS] モデル呼び出し #{self.call_count}")
        return None

    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if self.verbose:
            last = state["messages"][-1]
            print(f"📤 [CLASS] モデル応答: {str(last.content)[:60]}")
        return None

    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        print(f"🔴 [CLASS] エージェント完了 (モデル呼び出し回数: {self.call_count})")
        return None


# クラスのインスタンスを渡す
agent_7 = create_agent(
    model=MODEL_NAME,
    tools=[get_weather],
    middleware=[FullLoggingMiddleware(verbose=True)],
)

print("=" * 60)
print("🚀 クラス形式のミドルウェア")
print("=" * 60)
result = agent_7.invoke({"messages": [HumanMessage("沖縄の天気は？")]})
print("\n✅ 最終応答:", result["messages"][-1].content)

🚀 クラス形式のミドルウェア
🟢 [CLASS] エージェント開始
📥 [CLASS] モデル呼び出し #1
📤 [CLASS] モデル応答: 
📥 [CLASS] モデル呼び出し #2
📤 [CLASS] モデル応答: 沖縄の天気は雪で、気温は11°Cです。珍しい天気ですね。ほかに知りたい都市の天気はありますか？
🔴 [CLASS] エージェント完了 (モデル呼び出し回数: 2)

✅ 最終応答: 沖縄の天気は雪で、気温は11°Cです。珍しい天気ですね。ほかに知りたい都市の天気はありますか？


---
## 6. カスタム State

ミドルウェア専用のステートフィールドを追加できます。  
モデル呼び出し回数の追跡や、ユーザー情報の引き回しに便利です。

In [29]:
from typing_extensions import NotRequired


# ============================================
# カスタム State の定義
# AgentState を継承して独自フィールドを追加
# ============================================
class TrackingState(AgentState):
    """モデル呼び出し回数を追跡するカスタムState。"""
    model_call_count: NotRequired[int]  # NotRequired = 初期値なしでもOK


# --- デコレータ形式で state_schema を指定 ---
@before_model(state_schema=TrackingState, can_jump_to=["end"])
def check_call_limit(state: TrackingState, runtime: Runtime) -> dict[str, Any] | None:
    """モデル呼び出し回数が上限を超えたら終了。"""
    count = state.get("model_call_count", 0)
    print(f"📊 [カスタムState] 現在の呼び出し回数: {count}")
    if count >= 5:  # 5回以上で強制終了
        from langchain.messages import AIMessage
        print("🛑 上限に達しました! エージェントを終了します")
        return {
            "messages": [AIMessage("呼び出し上限に達しました。")],
            "jump_to": "end"
        }
    return None


@after_model(state_schema=TrackingState)
def increment_counter(state: TrackingState, runtime: Runtime) -> dict[str, Any] | None:
    """モデル呼び出し後にカウンターを+1。"""
    new_count = state.get("model_call_count", 0) + 1
    print(f"📊 [カスタムState] カウンター更新: {new_count}")
    return {"model_call_count": new_count}


agent_8 = create_agent(
    model=MODEL_NAME,
    tools=[get_weather, calculator],
    middleware=[check_call_limit, increment_counter],
)

print("=" * 60)
print("🚀 カスタムState でモデル呼び出し回数を追跡")
print("=" * 60)
result = agent_8.invoke({
    "messages": [HumanMessage("東京、大阪、名古屋の天気を全部調べて")],
    "model_call_count": 0,  # 初期値を渡す
})
print("\n✅ 最終応答:", result["messages"][-1].content)

🚀 カスタムState でモデル呼び出し回数を追跡
📊 [カスタムState] 現在の呼び出し回数: 0
📊 [カスタムState] カウンター更新: 1
📊 [カスタムState] 現在の呼び出し回数: 1
📊 [カスタムState] カウンター更新: 2

✅ 最終応答: 東京の天気は晴れで、気温は30°Cです。
大阪の天気は雨で、気温は-4°Cです。
名古屋の天気は雨で、気温は35°Cです。


---
## 7. 実行順序

複数ミドルウェアを渡した場合の実行順序を確認しましょう。

```
middleware=[A, B, C] の場合:

before_* → A → B → C（登録順）
after_*  → C → B → A（逆順）
wrap_*   → A が B を包み、B が C を包む（ネスト）
```

In [32]:
# ============================================
# 実行順序の確認
# 3つのミドルウェアの呼び出し順を可視化する
# ============================================

class OrderCheckMiddleware(AgentMiddleware):
    def __init__(self, label: str):
        super().__init__()
        self._label = label  # ← _name ではなく _label を使う

    @property
    def name(self) -> str:
        return f"OrderCheck_{self._label}"

    def before_agent(self, state, runtime):
        print(f"  before_agent  : {self._label}")
        return None

    def before_model(self, state, runtime):
        print(f"  before_model  : {self._label}")
        return None

    def after_model(self, state, runtime):
        print(f"  after_model   : {self._label}")
        return None

    def after_agent(self, state, runtime):
        print(f"  after_agent   : {self._label}")
        return None

    def wrap_model_call(self, request, handler):
        print(f"  wrap_model ▶  : {self._label}")
        result = handler(request)
        print(f"  wrap_model ◀  : {self._label}")
        return result

agent_order = create_agent(
    model=MODEL_NAME,
    tools=[],  # ツールなし = ループ1回で終了
    middleware=[
        OrderCheckMiddleware("A (1番目)"),
        OrderCheckMiddleware("B (2番目)"),
        OrderCheckMiddleware("C (3番目)"),
    ],
)

print("=" * 60)
print("🚀 実行順序の確認 (middleware=[A, B, C])")
print("=" * 60)
result = agent_order.invoke({"messages": [HumanMessage("こんにちは")]})
print("\n✅ 完了")

🚀 実行順序の確認 (middleware=[A, B, C])
  before_agent  : A (1番目)
  before_agent  : B (2番目)
  before_agent  : C (3番目)
  before_model  : A (1番目)
  before_model  : B (2番目)
  before_model  : C (3番目)
  wrap_model ▶  : A (1番目)
  wrap_model ▶  : B (2番目)
  wrap_model ▶  : C (3番目)
  wrap_model ◀  : C (3番目)
  wrap_model ◀  : B (2番目)
  wrap_model ◀  : A (1番目)
  after_model   : C (3番目)
  after_model   : B (2番目)
  after_model   : A (1番目)
  after_agent   : C (3番目)
  after_agent   : B (2番目)
  after_agent   : A (1番目)

✅ 完了


---
## 8. Agent Jumps — `jump_to` でフロー制御

ミドルウェアから `jump_to` を返すことで、エージェントの実行フローを変更できます。

| jump_to 値 | 効果 |
|---|---|
| `"end"` | エージェントを即座に終了 |
| `"tools"` | ツールノードにジャンプ |
| `"model"` | モデルノードにジャンプ |

In [34]:
from langchain.agents.middleware import hook_config
from langchain.messages import AIMessage


# ============================================
# メッセージ数の上限チェック → 上限超えたら終了
# ============================================
class MessageGuardMiddleware(AgentMiddleware):
    """メッセージ数が上限を超えたらエージェントを終了させる。"""

    def __init__(self, max_messages: int = 10):
        super().__init__()
        self.max_messages = max_messages

    @hook_config(can_jump_to=["end"])
    def before_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        msg_count = len(state["messages"])
        print(f"🛡️ [MessageGuard] メッセージ数: {msg_count}/{self.max_messages}")

        if msg_count >= self.max_messages:
            print("🛑 メッセージ上限に達しました! 強制終了します")
            return {
                "messages": [AIMessage("メッセージの上限に達したため、会話を終了します。")],
                "jump_to": "end",
            }
        return None


# ============================================
# NGワードチェック → 検出したら終了
# ============================================
class ContentGuardMiddleware(AgentMiddleware):
    """モデル応答にNGワードが含まれていたら終了させる。"""

    def __init__(self, blocked_words: list[str]):
        super().__init__()
        self.blocked_words = blocked_words

    @hook_config(can_jump_to=["end"])
    def after_model(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        last_content = str(state["messages"][-1].content).lower()
        for word in self.blocked_words:
            if word.lower() in last_content:
                print(f"⚠️ [ContentGuard] NGワード検出: '{word}'")
                return {
                    "messages": [AIMessage("申し訳ございません。その内容にはお答えできません。")],
                    "jump_to": "end",
                }
        return None


agent_9 = create_agent(
    model=MODEL_NAME,
    tools=[get_weather],
    middleware=[
        MessageGuardMiddleware(max_messages=10),
        ContentGuardMiddleware(blocked_words=["BLOCKED"]),
    ],
)

print("=" * 60)
print("🚀 jump_to によるフロー制御")
print("=" * 60)
result = agent_9.invoke({"messages": [HumanMessage("仙台の天気を教えて")]})
print("\n✅ 最終応答:", result["messages"][-1].content)

🚀 jump_to によるフロー制御
🛡️ [MessageGuard] メッセージ数: 1/10
🛡️ [MessageGuard] メッセージ数: 3/10

✅ 最終応答: 仙台の天気は現在雪で、気温は26°Cです。ほかに知りたいことはありますか？


---
## 9. `@dynamic_prompt` — 動的にシステムプロンプトを生成

実行時の状況（時刻、ユーザー情報、Stateの値など）に応じて  
システムプロンプトを動的に組み立てます。

In [36]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest
from datetime import datetime


@dynamic_prompt
def time_aware_prompt(request: ModelRequest) -> str:
    """現在時刻を含むシステムプロンプトを生成。"""
    now = datetime.now().strftime("%Y年%m月%d日 %H:%M")
    msg_count = len(request.messages)
    return (
        f"あなたは親切な日本語アシスタントです。\n"
        f"現在時刻: {now}\n"
        f"現在のメッセージ数: {msg_count}\n"
        f"回答は簡潔に、日本語で行ってください。"
    )


agent_10 = create_agent(
    model=MODEL_NAME,
    tools=[get_weather],
    middleware=[time_aware_prompt],
)

print("=" * 60)
print("🚀 dynamic_prompt で時刻入りプロンプト")
print("=" * 60)
result = agent_10.invoke({"messages": [HumanMessage("今何時？それと横浜の天気も")]})
print("\n✅ 最終応答:", result["messages"][-1].content)

🚀 dynamic_prompt で時刻入りプロンプト

✅ 最終応答: 現在の時刻は2026年4月8日 04時11分です。
横浜の天気は晴れで、気温は9℃です。


---
## 10. `wrap_model_call` 応用

### 10-1. 動的モデル切替
### 10-2. システムメッセージの加工

In [38]:
from langchain.chat_models import init_chat_model


# ============================================
# 10-1. 動的モデル切替
# 会話が長くなったら高性能モデルに切替
# ============================================
@wrap_model_call
def dynamic_model_switch(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """メッセージ数に応じてモデルを切り替える。"""
    msg_count = len(request.messages)

    if msg_count > 10:
        # 長い会話 → 高性能モデル
        model = init_chat_model(MODEL_NAME)
        print(f"🔄 [モデル切替] 高性能モデル使用 (メッセージ数: {msg_count})")
    else:
        # 短い会話 → 軽量モデル
        model = init_chat_model(MODEL_NAME_CHEAP)
        print(f"🔄 [モデル切替] 軽量モデル使用 (メッセージ数: {msg_count})")

    return handler(request.override(model=model))


agent_11 = create_agent(
    model=MODEL_NAME,  # デフォルトモデル
    tools=[get_weather],
    middleware=[dynamic_model_switch],
)

print("=" * 60)
print("🚀 動的モデル切替")
print("=" * 60)
#result = agent_11.invoke({"messages": [HumanMessage("広島の天気は？")]})
result = agent_11.invoke({"messages": [HumanMessage("広島の天気は？そのとAI Agentとはなにかを簡単に説明してください。")]})
print("\n✅ 最終応答:", result["messages"][-1].content)

🚀 動的モデル切替
🔄 [モデル切替] 軽量モデル使用 (メッセージ数: 1)
🔄 [モデル切替] 軽量モデル使用 (メッセージ数: 4)

✅ 最終応答: 広島の天気は曇りで、気温は28°Cです。また、AI Agentの天気は晴れで、気温も28°Cです。AI Agentとは、人工知能を活用したソフトウェアやシステムのことで、ユーザーの質問や依頼に対して自動的に対応し、情報提供やタスクの実行を行うものです。


In [39]:
from langchain.messages import SystemMessage


# ============================================
# 10-2. システムメッセージの加工
# 実行時にコンテキストを追加注入する
# ============================================
@wrap_model_call
def inject_context(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """システムメッセージに追加コンテキストを注入。"""
    # 既存のシステムメッセージの内容を取得
    existing_blocks = list(request.system_message.content_blocks)

    # 新しいコンテキストを追加
    additional_context = {
        "type": "text",
        "text": (
            "\n\n【追加情報】\n"
            "- ユーザーは日本在住です\n"
            "- 気温は摂氏で回答してください\n"
            "- 回答は丁寧語で行ってください"
        )
    }
    new_content = existing_blocks + [additional_context]
    new_system = SystemMessage(content=new_content)

    print("💉 [inject_context] システムメッセージにコンテキスト追加完了")
    return handler(request.override(system_message=new_system))


agent_12 = create_agent(
    model=MODEL_NAME,
    tools=[get_weather],
    system_prompt="あなたは天気情報アシスタントです。",
    middleware=[inject_context],
)

print("=" * 60)
print("🚀 システムメッセージへのコンテキスト注入")
print("=" * 60)
result = agent_12.invoke({"messages": [HumanMessage("神戸の天気は？")]})
print("\n✅ 最終応答:", result["messages"][-1].content)

🚀 システムメッセージへのコンテキスト注入
💉 [inject_context] システムメッセージにコンテキスト追加完了
💉 [inject_context] システムメッセージにコンテキスト追加完了

✅ 最終応答: 神戸の現在の天気は雪で、気温は摂氏35度です。何か他に知りたいことはございますか？


---
## 11. ビルトインミドルウェア

LangChain には予め用意されたミドルウェアがあります。  
自分で書かなくても `import` するだけで使えます。

### 11-1. `SummarizationMiddleware` — 自動要約

会話が長くなるとトークン上限に達します。  
このミドルウェアは閾値を超えたら古いメッセージを自動要約します。

In [40]:
from langchain.agents.middleware import SummarizationMiddleware


# ============================================
# SummarizationMiddleware
# trigger: 要約を開始する条件
# keep: 要約せずに残すメッセージ数
# ============================================
agent_summarize = create_agent(
    model=MODEL_NAME,
    tools=[get_weather],
    middleware=[
        SummarizationMiddleware(
            model=MODEL_NAME_CHEAP,         # 要約に使うモデル（安価なもの推奨）
            trigger=("tokens", 4000),        # 4000トークンを超えたら要約開始
            keep=("messages", 20),           # 最新20メッセージは残す
        ),
    ],
)

print("✅ SummarizationMiddleware 設定完了")
print("   4000トークン超過時に自動要約、最新20メッセージは保持")

# デモ実行
result = agent_summarize.invoke({
    "messages": [HumanMessage("東京の天気を教えてください")]
})
print("\n✅ 最終応答:", result["messages"][-1].content)

✅ SummarizationMiddleware 設定完了
   4000トークン超過時に自動要約、最新20メッセージは保持

✅ 最終応答: 東京の天気は曇りで、気温は-4°Cです。


### 11-2. `HumanInTheLoopMiddleware` — 人間の承認

危険なツール呼び出し前に **人間の承認** を求めます。  
⚠️ `checkpointer` が必要です（interrupt で実行を中断するため）。

In [41]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


# ============================================
# HumanInTheLoopMiddleware
# 特定ツールの実行前に人間の承認を求める
# ============================================

@tool
def delete_data(target: str) -> str:
    """データを削除します（危険な操作）。"""
    return f"✅ {target} を削除しました"


agent_hitl = create_agent(
    model=MODEL_NAME,
    tools=[get_weather, delete_data],
    checkpointer=InMemorySaver(),  # 必須! interrupt で状態を保存するため
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "delete_data": {           # このツールは承認必要
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "get_weather": False,       # このツールは承認不要
            }
        ),
    ],
)

print("✅ HumanInTheLoopMiddleware 設定完了")
print("   delete_data → 承認必要")
print("   get_weather → 承認不要")
print()
print("💡 NOTE: 実際に interrupt を使うには stream() と")
print("   Command(resume=...) でやり取りします。")
print("   詳細は LangChain の Human-in-the-loop ドキュメントを参照してください。")

✅ HumanInTheLoopMiddleware 設定完了
   delete_data → 承認必要
   get_weather → 承認不要

💡 NOTE: 実際に interrupt を使うには stream() と
   Command(resume=...) でやり取りします。
   詳細は LangChain の Human-in-the-loop ドキュメントを参照してください。


### 11-3. その他のビルトインミドルウェア一覧

以下は `langchain.agents.middleware` から import できるビルトインです:

| ミドルウェア | 説明 |
|---|---|
| `SummarizationMiddleware` | トークン上限接近時に古いメッセージを自動要約 |
| `HumanInTheLoopMiddleware` | 危険なツール実行前に人間の承認を要求 |
| `ModelCallLimitMiddleware` | モデル呼び出し回数の上限を設定 |
| `ToolCallLimitMiddleware` | ツール実行回数の上限を設定 |
| `ModelFallbackMiddleware` | エラー時に代替モデルへ自動フォールバック |
| `PIIMiddleware` | 個人情報（PII）の検出と隠蔽 |
| `ToolRetryMiddleware` | ツール実行失敗時のバックオフ付きリトライ |
| `ToolSelectionMiddleware` | LLMで関連ツールを事前選択 |
| `LLMToolEmulator` | テスト用にツール実行をLLMでエミュレート |
| `ShellToolMiddleware` | 永続的シェルツールを登録 |
| `TodoMiddleware` | ToDoリスト管理機能を提供 |

---
## 12. 総合例 — 本番想定パターン

複数のミドルウェアを組み合わせた実践的な構成例です。

In [42]:
import json
from datetime import datetime


# ============================================
# 総合ミドルウェア1: 全体ロギング（クラス形式）
# ============================================
class ProductionLoggingMiddleware(AgentMiddleware):
    """本番用のロギングミドルウェア。
    全ライフサイクルイベントを構造化ログで出力する。
    """

    def __init__(self, agent_name: str = "default"):
        super().__init__()
        self.agent_name = agent_name
        self.start_time = None

    def _log(self, event: str, data: dict):
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "agent": self.agent_name,
            "event": event,
            **data,
        }
        print(f"📋 LOG: {json.dumps(log_entry, ensure_ascii=False)}")

    def before_agent(self, state, runtime):
        self.start_time = time.time()
        self._log("agent_start", {"message_count": len(state["messages"])})
        return None

    def before_model(self, state, runtime):
        self._log("model_call_start", {"message_count": len(state["messages"])})
        return None

    def after_model(self, state, runtime):
        last = state["messages"][-1]
        has_tools = hasattr(last, "tool_calls") and bool(last.tool_calls)
        self._log("model_call_end", {"has_tool_calls": has_tools})
        return None

    def after_agent(self, state, runtime):
        elapsed = time.time() - self.start_time if self.start_time else 0
        self._log("agent_end", {
            "total_messages": len(state["messages"]),
            "elapsed_seconds": round(elapsed, 2),
        })
        return None


# ============================================
# 総合ミドルウェア2: モデル呼び出し計測（デコレータ形式）
# ============================================
@wrap_model_call
def production_retry(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """本番用リトライ + 計測。"""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            start = time.time()
            response = handler(request)
            elapsed = time.time() - start
            print(f"⚡ モデル応答: {elapsed:.2f}秒")
            return response
        except Exception as e:
            print(f"⚠️ リトライ {attempt + 1}/{max_retries}: {e}")
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)  # 指数バックオフ


# ============================================
# 総合ミドルウェア3: ツール監視（デコレータ形式）
# ============================================
@wrap_tool_call
def production_tool_monitor(
    request: ToolCallRequest,
    handler: Callable[[ToolCallRequest], ToolMessage | Command],
) -> ToolMessage | Command:
    """本番用ツール監視。"""
    name = request.tool_call["name"]
    start = time.time()
    try:
        result = handler(request)
        elapsed = time.time() - start
        print(f"🔧 ツール '{name}' 成功 ({elapsed:.3f}秒)")
        return result
    except Exception as e:
        elapsed = time.time() - start
        print(f"❌ ツール '{name}' 失敗 ({elapsed:.3f}秒): {e}")
        raise


# ============================================
# 本番エージェントの組み立て
# ============================================
production_agent = create_agent(
    model=MODEL_NAME,
    system_prompt=(
        "あなたは多機能アシスタントです。"
        "天気の確認、計算、Web検索ができます。"
        "日本語で簡潔に回答してください。"
    ),
    tools=ALL_TOOLS,
    middleware=[
        # 1. ロギング（最初 = before は最初に、after は最後に実行）
        ProductionLoggingMiddleware(agent_name="production-v1"),
        # 2. リトライ + 計測
        production_retry,
        # 3. ツール監視
        production_tool_monitor,
    ],
)

print("=" * 60)
print("🚀 本番想定: ロギング + リトライ + ツール監視")
print("=" * 60)
result = production_agent.invoke({
    "messages": [HumanMessage("東京の天気と、999 * 123 の計算結果を教えて")]
})
print("\n" + "=" * 60)
print("✅ 最終応答:")
print(result["messages"][-1].content)

🚀 本番想定: ロギング + リトライ + ツール監視
📋 LOG: {"timestamp": "2026-04-08T04:17:55.584945", "agent": "production-v1", "event": "agent_start", "message_count": 1}
📋 LOG: {"timestamp": "2026-04-08T04:17:55.585374", "agent": "production-v1", "event": "model_call_start", "message_count": 1}
⚡ モデル応答: 1.28秒
📋 LOG: {"timestamp": "2026-04-08T04:17:56.870975", "agent": "production-v1", "event": "model_call_end", "has_tool_calls": true}
🔧 ツール 'get_weather' 成功 (0.001秒)
🔧 ツール 'calculator' 成功 (0.001秒)
📋 LOG: {"timestamp": "2026-04-08T04:17:56.877359", "agent": "production-v1", "event": "model_call_start", "message_count": 4}
⚡ モデル応答: 0.89秒
📋 LOG: {"timestamp": "2026-04-08T04:17:57.765405", "agent": "production-v1", "event": "model_call_end", "has_tool_calls": false}
📋 LOG: {"timestamp": "2026-04-08T04:17:57.765914", "agent": "production-v1", "event": "agent_end", "total_messages": 5, "elapsed_seconds": 2.18}

✅ 最終応答:
東京の天気は晴れで、気温は21°Cです。計算結果は999 × 123 = 122,877です。


---
## まとめ

### フック一覧

| フック | タイプ | タイミング | 典型的な用途 |
|---|---|---|---|
| `before_agent` | ノード | エージェント開始前（1回） | 初期化、リソース準備 |
| `before_model` | ノード | モデル呼び出し前（毎回） | ロギング、バリデーション |
| `after_model` | ノード | モデル応答後（毎回） | 応答チェック、ガードレール |
| `after_agent` | ノード | エージェント完了後（1回） | クリーンアップ、集計 |
| `wrap_model_call` | ラップ | モデル呼び出しを包む | リトライ、キャッシュ、モデル切替 |
| `wrap_tool_call` | ラップ | ツール実行を包む | 監視、リトライ、セキュリティ |
| `dynamic_prompt` | 便利 | モデル呼び出し前 | 動的プロンプト生成 |

### ベストプラクティス

1. **単一責任** — 1つのミドルウェアは1つの仕事だけ
2. **順序を意識** — `before_*` は登録順、`after_*` は逆順
3. **エラーハンドリング** — ミドルウェアの例外でエージェントが止まらないように
4. **ビルトインを活用** — 自作の前にビルトインをチェック
5. **カスタムState は NotRequired** — 初期値がなくても動くように

---

🎉 **以上で全パターンのカバーが完了です！**

質問やフィードバックがあればお気軽にどうぞ。